# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanaanwar25/flyrank-ml-internship-sana/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os
import subprocess

REPO_URL = "https://github.com/sanaanwar25/flyrank-ml-internship-sana"
REPO_DIR = "/content/flyrank-ml-internship-sana"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())
print("Data file exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Current folder: /content/flyrank-ml-internship-sana
Data file exists: True


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item for one anonymized client (client_hash_id × content_hash_id). February 2026 (2026-02-01 to 2026-02-28) is the feature window, and March 2026 (2026-03-01 to 2026-03-31) is the label window. The windows do not overlap, so March outcomes are not used to create February features.

In [8]:
print("Unit: client_hash_id × content_hash_id")
print("Feature window: 2026-02-01 to 2026-02-28")
print("Label window: 2026-03-01 to 2026-03-31")

Unit: client_hash_id × content_hash_id
Feature window: 2026-02-01 to 2026-02-28
Label window: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: February search/performance fields that are known before the prediction cutoff.
Label: March outcome fields, such as March clicks and impressions used to define the outcome.
Context: client_hash_id, content_hash_id, and descriptive content fields used to identify or describe a row.
Excluded: trend_direction and trend_pct because they directly define the declining label and would cause target leakage. March outcome fields are also excluded from features because they occur after the prediction cutoff.

In [9]:
print("Excluded from model features:")
print("- trend_direction: defines the label")
print("- trend_pct: directly describes the target outcome")
print("- March outcome fields: occur after the feature cutoff")

Excluded from model features:
- trend_direction: defines the label
- trend_pct: directly describes the target outcome
- March outcome fields: occur after the feature cutoff


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\nColumns:")
print(df.columns.tolist())


Rows: 30000
Columns: 44

Duplicate rows: 0

Missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
dtype: int64

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_positio

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limits: The data does not provide a complete causal history of why a page changed. Feature and label windows are separated, but they represent limited time periods. Some history may be unbalanced across pages, and GSC availability may differ between records. Window-based measurements can also overlap with longer-term page changes that are not observed here. Therefore, results will be treated as observed, measured, directional decision-support rather than causal proof.

In [11]:
print("Data limits checked:")
print("1. Limited observation window")
print("2. GSC availability can vary")
print("3. Feature and label windows must remain separate")
print("4. Results are directional, not causal")

Data limits checked:
1. Limited observation window
2. GSC availability can vary
3. Feature and label windows must remain separate
4. Results are directional, not causal


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.